# NF-ToN-IoT-v3 — Phase 3: data preprocessing (cleaning)

**Owner:** Cyrine Chabbar (cleaning code prepared by Ahmed Hamouda with the same steps as the other datasets). Applies the
approved decisions of phase 2 for this dataset (`report/data_problems/phase2_data_problems.pdf`, section 3 and section 6).
Phase 2 evidence: `03_ton.ipynb`. The result has exactly the same 56 columns as `UNSW_clean.parquet` and `CIC18_clean.parquet`.

| Step | What | Decision |
|---|---|---|
| 0 | Load the raw CSV | – |
| 1 | Drop the two `..._SECOND_BYTES` columns | T3 |
| 2 | `L7_PROTO` → `L7_MAIN` + `L7_SUB` (= 0, no decimals in this file) | T3 |
| 3 | Remove exact duplicate rows | T4 |
| 4 | Check for flows that carry two labels (the C5 rule; phase 2 found none here) | C5 rule |
| 5 | Remove XSS and injection flows to the DNS port 53 | T5 |
| 6 | One row per (behaviour, label), with a `count` column | Section 6 |
| 7 | Shared `clean()`: drop the IP columns, NaN / inf, duplicates | M3 + shared rule |
| 8 | `map_labels()` (lower-case labels get clean names) and drop `Label` | T1 |
| 9 | Checks and save | – |

**Not applied here, on purpose:** the CIC-2018 rule C7. ToN-IoT's attack flows that look like Benign are kept (T6): the
flows are too short to show the difference, the labels are not wrong. **Also kept:** attack-vs-attack conflicts (T7), MITM
and Ransomware (T8), port 0, TTL = 0, outliers. **After the merge:** M1, M6, M7, M8, M9.

27.5 million rows (the largest file): about 10 GB of memory; close other heavy programs before running.

## Step 0 — Load the raw data

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.parquet as pq

from src.config import *

pd.set_option("display.max_columns", 100)
SOURCE = "ToN"
path = RAW_DIR / DATASETS[SOURCE]["file"]

header = pd.read_csv(path, nrows=0).columns
# int32 halves the memory; pyarrow stops with an error if a value does not fit. L7_PROTO as an integer also proves
# there are no decimal sub-codes in this file.
types = {c: pa.int32() for c in header if c not in ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Attack"]}
for c in ["FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "DNS_TTL_ANSWER"]:
    types[c] = pa.int64()
table = pv.read_csv(path, convert_options=pv.ConvertOptions(column_types=types))
# Convert one column at a time and free it from the arrow table: the peak is the table + one column,
# instead of the table + the whole DataFrame (about 20 GB for this file).
data = {}
for c in list(table.column_names):
    data[c] = table.column(c).to_pandas()
    table = table.drop_columns([c])
del table
df = pd.DataFrame(data, copy=False)   # copy=False: no extra 5 GB copy of the numeric columns
del data

raw_labels = df["Attack"].value_counts()
steps = []
keep = np.ones(len(df), dtype=bool)   # steps 3-5 mark removals here; step 6 keeps the marked rows in one go
def log(step, decision, what, rows=None):
    rows = len(df) if rows is None else int(rows)
    steps.append({"step": step, "decision": decision, "rows": rows, "columns": df.shape[1], "what": what})
    print(f"{step}: {rows:,} rows x {df.shape[1]} columns  ({what})")

def removed(mask, name):
    """Show what a removal takes away, per label (compared with the rows still kept before it)."""
    print(f"{name}: {int(mask.sum()):,} rows")
    if mask.any():
        attack = df["Attack"].to_numpy()
        before = pd.Series(attack[keep]).value_counts().rename("before")
        display(pd.Series(attack[mask]).value_counts().rename("removed").to_frame()
                .join(before).assign(after=lambda t: t["before"] - t["removed"]))

log("0 load", "-", "raw file")
raw_labels

0 load: 27,520,260 rows x 55 columns  (raw file)


Attack
Benign        16792214
ddos           4141256
xss            2834435
password       1594777
scanning       1358977
injection       381777
dos             203456
Backdoor        203384
mitm              6013
ransomware        3971
Name: count, dtype: int64

## Step 1 — Drop the two `..._SECOND_BYTES` columns (T3)
Integers without NaN here, but with the same wrong meaning as in the other datasets (bytes per millisecond, directions
swapped); dropped everywhere so the four cleaned files have the same columns.

In [2]:
df = df.drop(columns=["SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES"])
log("1 drop SECOND_BYTES", "T3", "2 columns removed")

1 drop SECOND_BYTES: 27,520,260 rows x 53 columns  (2 columns removed)


## Step 2 — `L7_PROTO` → `L7_MAIN` + `L7_SUB` (T3)
No sub-codes in ToN-IoT, so `L7_MAIN` = `L7_PROTO` and `L7_SUB` = 0, the same two columns as the other cleaned files.

In [3]:
pos = df.columns.get_loc("L7_PROTO")
df.insert(pos, "L7_MAIN", df["L7_PROTO"].astype("int32"))
df.insert(pos + 1, "L7_SUB", np.zeros(len(df), dtype="int32"))
df = df.drop(columns="L7_PROTO")
log("2 split L7_PROTO", "T3", "L7_PROTO -> L7_MAIN + L7_SUB = 0")

2 split L7_PROTO: 27,520,260 rows x 54 columns  (L7_PROTO -> L7_MAIN + L7_SUB = 0)


## Step 3 — Remove exact duplicate rows (T4)
Rows identical in every column, including IPs and the millisecond timestamp. One Benign flow appears hundreds of times and
every Backdoor flow twice. Compared through a 64-bit hash of each row (same result as `df.duplicated()`, far less memory).

In [4]:
row_hash = pd.util.hash_pandas_object(df, index=False).to_numpy()
dup = pd.Series(row_hash).duplicated().to_numpy()
removed(dup, "exact duplicates")
keep &= ~dup
del row_hash
log("3 exact duplicates", "T4", f"{int(dup.sum()):,} copies removed", keep.sum())

exact duplicates: 1,816,137 rows


,removed,before,after
Benign,1713700,16792214,15078514
Backdoor,101692,203384,101692
mitm,745,6013,5268


3 exact duplicates: 25,704,123 rows x 54 columns  (1,816,137 copies removed)


## Step 4 — Check for flows that carry two labels (C5 rule)
The same flow (identical in every column except the labels) with two different labels would be removed entirely, as in
CIC-2018. Phase 2 found none in ToN-IoT; this confirms it on the cleaned rows.

In [5]:
feat = [c for c in df.columns if c not in ("Label", "Attack")]
fh = pd.util.hash_pandas_object(df[feat], index=False).to_numpy()
n_labels = pd.Series(df["Attack"].to_numpy()[keep]).groupby(fh[keep]).nunique()
two = keep & np.isin(fh, n_labels.index[n_labels > 1])
removed(two, "flows with two labels")
keep &= ~two
del fh
log("4 two labels (check)", "C5 rule", f"{int(two.sum()):,} rows removed", keep.sum())

flows with two labels: 0 rows
4 two labels (check): 25,704,123 rows x 54 columns  (0 rows removed)


## Step 5 — Remove web attacks labelled on DNS traffic (T5)
XSS and injection travel inside HTTP requests over TCP (ports 80 / 443). The XSS / injection flows to port 53 are 100% UDP,
recognised by nDPI as DNS, one packet of about 65 bytes: the attacker machine's background DNS lookups labelled with the
attack. DoS and DDoS flows to port 53 are kept (a DNS flood is a real DoS attack).

In [6]:
dns_web = keep & (df["Attack"].isin(["xss", "injection"]) & (df["L4_DST_PORT"] == 53)).to_numpy()
print("protocol of these flows:", dict(df.loc[dns_web, "PROTOCOL"].map({6: "TCP", 17: "UDP"}).value_counts()))
removed(dns_web, "XSS / injection to port 53")
keep &= ~dns_web
log("5 web attacks on DNS port", "T5", f"{int(dns_web.sum()):,} rows removed", keep.sum())

protocol of these flows: {'UDP': np.int64(1261353)}
XSS / injection to port 53: 1,261,353 rows


,removed,before,after
xss,1244884,2834435,1589551
injection,16469,381777,365308


5 web attacks on DNS port: 24,442,770 rows x 54 columns  (1,261,353 rows removed)


## Step 6 — One row per (behaviour, label), with `count` (section 6)
Repeats of the same behaviour with the same label become one row; `count` keeps how many flows it stands for. Runs before
the IPs are dropped, so different machines doing the same thing are counted as separate flows. The attack flows that look
like Benign (T6) are kept as their own rows with their attack label.

In [7]:
flows_before = int(keep.sum())
df = collapse_behaviours(df, "Attack", keep=keep)   # only the rows kept by steps 3-5
del keep
print(f"{flows_before:,} flows -> {len(df):,} rows; sum of count = {df['count'].sum():,}")
assert df["count"].sum() == flows_before
display(pd.DataFrame({"raw flows": raw_labels, "rows kept": df["Attack"].value_counts(),
                      "largest count": df.groupby("Attack")["count"].max()}).fillna(0).astype(int).sort_values("raw flows", ascending=False))
log("6 collapse behaviours", "section 6", "repeats -> one row + count")

24,442,770 flows -> 5,637,642 rows; sum of count = 24,442,770


,raw flows,rows kept,largest count
Attack,,,
Benign,16792214,2282642,102957
ddos,4141256,451474,1671924
xss,2834435,1497451,6400
password,1594777,697124,54266
scanning,1358977,225074,265628
injection,381777,333652,2344
dos,203456,121803,2209
Backdoor,203384,20641,3532
mitm,6013,4951,25


6 collapse behaviours: 5,637,642 rows x 55 columns  (repeats -> one row + count)


## Step 7 — Shared `clean()` (`src/config.py`)

In [8]:
rows_before = len(df)
df, stats = clean(df, SOURCE)
print("rows lost in clean():", rows_before - len(df))
log("7 clean()", "M3 + shared rule", f"IP columns dropped; {rows_before - len(df)} rows lost")
stats

rows lost in clean(): 0
7 clean(): 5,637,642 rows x 53 columns  (IP columns dropped; 0 rows lost)


{'source_dataset': 'ToN',
 'rows_raw': 5637642,
 'cols_raw': 55,
 'id_columns_dropped': 'IPV4_SRC_ADDR, IPV4_DST_ADDR',
 'inf_values': 0,
 'nan_values': 0,
 'imputed_columns': 'none',
 'rows_with_nan': 0,
 'duplicate_rows': 0,
 'rows_clean': 5637642,
 'cols_clean': 53}

## Step 8 — Shared labels (`map_labels`) and drop `Label` (T1)
ToN-IoT writes most labels in lower case (`ddos`, `xss`, `password`…); the taxonomy lookup ignores case, so they get the
same clean names as in the other datasets (DDoS, XSS, Password…).

In [9]:
df = map_labels(df, "Attack", SOURCE)
assert (df["is_attack"].astype(int) == df["Label"].astype(int)).all(), "Label and is_attack disagree"
df = df.drop(columns="Label")
display(df.groupby(["attack_family", "attack_label"], observed=True)["count"].agg(rows="size", flows="sum"))
log("8 map_labels + drop Label", "T1", "4 label columns added, Label dropped")

rows     flows
attack_family             attack_label                   
Benign                    Benign        2282642  15078514
Credential attacks        Password       697124   1594777
Denial of service         DDoS           451474   4141256
                          DoS            121803    203456
Interception and spoofing MITM             4951      5268
Malware and botnets       Backdoor        20641    101692
                          Ransomware       2830      3971
Reconnaissance            Scanning       225074   1358977
Web and injection         Injection      333652    365308
                          XSS           1497451   1589551

8 map_labels + drop Label: 5,637,642 rows x 56 columns  (4 label columns added, Label dropped)


## Step 9 — Checks and save

In [10]:
checks = validate(df)
print("validate(): total violations", int(checks["violations"].sum()))
num = df.select_dtypes("number")
print("NaN cells:", int(df.isna().sum().sum()), "| inf cells:", int(np.isinf(num).sum().sum()))
print("IP columns left:", [c for c in df.columns if "ADDR" in c])
ref = PROCESSED_DIR / "UNSW_clean.parquet"
print("same columns and order as UNSW_clean:", pq.read_schema(ref).names == list(df.columns) if ref.exists() else "UNSW file not found")
print(f"final: {len(df):,} rows x {df.shape[1]} columns | flows represented {int(df['count'].sum()):,} of {int(raw_labels.sum()):,} raw")

validate(): total violations 0


NaN cells: 0 | inf cells: 0
IP columns left: []
same columns and order as UNSW_clean: True
final: 5,637,642 rows x 56 columns | flows represented 24,442,770 of 27,520,260 raw


In [11]:
log("9 final", "-", "cleaned table")
lab_map = map_labels(pd.DataFrame({"Attack": raw_labels.index}), "Attack", SOURCE).set_index("Attack")["attack_label"]
labels = cleaning_report(steps, raw_labels.groupby(lab_map).sum(), df, SOURCE)
out = save_clean(df, SOURCE)
print("saved", out.relative_to(ROOT), f"({out.stat().st_size / 1e6:.0f} MB)")
display(pd.DataFrame(steps))
labels.sort_values("flows_raw", ascending=False)

9 final: 5,637,642 rows x 56 columns  (cleaned table)


saved data\processed\ToN_clean.parquet (201 MB)


,step,decision,rows,columns,what
0,0 load,-,27520260,55,raw file
1,1 drop SECOND_BYTES,T3,27520260,53,2 columns removed
2,2 split L7_PROTO,T3,27520260,54,L7_PROTO -> L7_MAIN + L7_SUB = 0
3,3 exact duplicates,T4,25704123,54,"1,816,137 copies removed"
4,4 two labels (check),C5 rule,25704123,54,0 rows removed
5,5 web attacks on DNS port,T5,24442770,54,"1,261,353 rows removed"
6,6 collapse behaviours,section 6,5637642,55,repeats -> one row + count
7,7 clean(),M3 + shared rule,5637642,53,IP columns dropped; 0 rows lost
8,8 map_labels + drop Label,T1,5637642,56,"4 label columns added, Label dropped"
9,9 final,-,5637642,56,cleaned table


,source_dataset,flows_raw,rows_clean,flows_represented,attack_family
attack_label,,,,,
Benign,ToN,16792214,2282642,15078514,Benign
DDoS,ToN,4141256,451474,4141256,Denial of service
XSS,ToN,2834435,1497451,1589551,Web and injection
Password,ToN,1594777,697124,1594777,Credential attacks
Scanning,ToN,1358977,225074,1358977,Reconnaissance
Injection,ToN,381777,333652,365308,Web and injection
DoS,ToN,203456,121803,203456,Denial of service
Backdoor,ToN,203384,20641,101692,Malware and botnets
MITM,ToN,6013,4951,5268,Interception and spoofing
